# 2 · Building my first simple agent

**Goal:** an agent that can **list files**, **read them** and **answer questions** about them, using MiniCPM (`openbmb/MiniCPM5-2B`).

**The agent loop**

```
┌─► 1. Construct prompt   (agent rules + memory)
│   2. Generate response  (LLM picks the next action)
│   3. Parse response     (extract tool name + arguments)
│   4. Execute action     (run the matching Python function)
│   5. Update memory      (store the action and its result)
└── 6. Continue?          (stop on terminate() or max iterations)
```

**Quasi-agent (notebook 1) vs agent**

| | Quasi-agent | Agent |
|---|---|---|
| Sequence of steps | hard-coded by me | decided by the LLM at each iteration |
| Tools | none | Python functions the LLM can call |
| Memory | conversation | conversation + results of actions |

📄 Full notes: [`agent_loop_notes.md`](agent_loop_notes.md)


## Setup

In [ ]:
import json                       # parse the agent's JSON actions
from typing import Dict, List     # type hints

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# Load MiniCPM (~5 GB, downloaded on first run, then cached)
model_id = "openbmb/MiniCPM5-2B"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",   # keep the precision the weights were saved in
    device_map="auto",    # use the GPU automatically when available
)

print("Model loaded successfully!")

In [ ]:
def generate_response(messages: List[Dict]) -> str:
    """Send a list of chat messages to MiniCPM and return its reply as text."""
    # Format the messages with the model's chat template -> token IDs
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,   # add the "assistant's turn" marker
        enable_thinking=False,        # skip MiniCPM's hidden reasoning step
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    # Generate (greedy, deterministic); no gradients needed
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
        )

    # Decode only the new tokens (drop the prompt)
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )

    return response

## Step 1 · Construct the prompt

```python
prompt = agent_rules + memory
```

| Part | What it is | Why |
|---|---|---|
| `agent_rules` | **system** message: behaviour, available tools, constraints, output format | the agent always knows *what it can do* and *how to answer* |
| `memory` | record of past interactions: user input, agent actions, results of actions | the agent knows *what just happened* and *what happened before* → informs the next action |

- Rebuilt at **every iteration** of the loop → the rules are never forgotten, the memory keeps growing.
- Memory also holds the **task description** (the user's request).

### Agent rules

The system message contains:

- **Role** → *"You are an AI agent that can perform tasks by using available tools."*
- **Tools** → each tool = a Python function in the code
  - `list_files()` → list files in the current directory
  - `read_file(file_name)` → read a file's content
  - `terminate(message)` → stop the loop and summarise for the user
- **Constraints** → e.g. *list files before reading them*; *every response MUST have an action*
- **Output format** → a fixed ```` ```action {...} ``` ```` JSON block → easy and reliable to parse

**Key idea: reasoning vs execution**

- The agent does **not** write the tools; it **orchestrates** them:
  - *which* tool to use,
  - *when* to use it,
  - *with which arguments*.
- The tools' logic is predefined in code → the LLM focuses on **what** to do, the code on **how**.
- The agent adapts step by step to new information (e.g. lists files → file exists? → read it, or report an error).
- No hard-coded workflow ("glue code") → the agent builds the sequence of actions itself → modular and extensible.

*Example:* user asks *"What files are here?"* → expected answer:
```action
{"tool_name": "list_files", "args": {}}
```

In [ ]:
# Agent rules: system message sent at the start of EVERY prompt
agent_rules = [{
    "role": "system",
    "content": """
You are an AI agent that can perform tasks by using available tools.

Available tools:
- list_files() -> List[str]: List all files in the current directory.
- read_file(file_name: str) -> str: Read the content of a file.
- terminate(message: str): End the agent loop and print a summary to the user.

If a user asks about files, list them before reading.

Every response MUST have an action.
Respond in this format:

```action
{
    "tool_name": "insert tool_name",
    "args": {...fill in any required arguments here...}
}
```
"""
}]

### Memory (example)

A conversation already one iteration in:

| # | Role | Content | Meaning |
|---|---|---|---|
| 1 | `user` | *What files are in this directory?* | the task |
| 2 | `assistant` | ```` ```action {"tool_name": "list_files", "args": {}} ``` ```` | the agent's action |
| 3 | `user` | `["file1.txt", "file2.txt"]` | the **result** of the action, fed back to the agent |

→ Results of actions come back as `user` messages, so the agent can decide its next step.

In [ ]:
# Memory: task + past actions + their results
memory = [
    {"role": "user", "content": "What files are in this directory?"},                              # task
    {"role": "assistant", "content": "```action\n{\"tool_name\":\"list_files\",\"args\":{}}\n```"},  # action
    {"role": "user", "content": "[\"file1.txt\", \"file2.txt\"]"},                                  # result
]

# Step 1: full prompt = rules first, then memory
prompt = agent_rules + memory

## Step 2 · Generate the response

```python
response = generate_response(prompt)
```

- Sends the prompt (rules + memory) to the LLM.
- The LLM **decides the next action** based on context and rules.
- Expected output: a structured ```` ```action {...} ``` ```` block → parsed and executed in the next steps.
- Here the LLM is MiniCPM run locally with `transformers` (the course uses the LiteLLM library to call hosted models; the idea is the same).

In [ ]:
# Step 2: ask the LLM for the next action
response = generate_response(prompt)
print(response)

## Interfacing the agent with the environment

```
LLM response ──► 3. parse ──► action {tool_name, args} ──► 4. execute ──► result ──► (5. feedback to memory)
```

- The LLM only produces **text** → the interface turns it into a real **action** in the environment.
- The interface:
  1. figures out **which action** the response corresponds to,
  2. **executes** it,
  3. later gives the agent **feedback** on the result.

## Step 3 · Parse the response

**Goal:** extract the intended action and its parameters from the LLM's output.

- Expected format: **JSON** inside a Markdown block marked ```` ```action ```` → no ambiguity.
- How: take the content between the ```` ```action ```` markers → `json.loads` → check it has `tool_name` and `args`.
- Output: a structured action:
  ```python
  {"tool_name": "list_files", "args": {}}
  ```
  → `tool_name` = **which** action, `args` = its **inputs**.

**Fallback if the response is not valid**

| Problem | Returned action |
|---|---|
| valid JSON, but no `tool_name` / `args` | `error`: *"You must respond with a JSON tool invocation."* |
| invalid JSON | `error`: *"Invalid JSON response. You must respond with a JSON tool invocation."* |

- The error is sent back to the LLM **as if it came from the `user`** → the LLM is asked to retry with a valid tool call.
- → the agent can **recover** when the LLM drifts from the expected format, instead of crashing.

In [ ]:
def extract_markdown_block(response: str, block_type: str = "json") -> str:
    """Return the content of the Markdown code block of type `block_type`.

    1. If the response contains a ```<block_type> block (e.g. ```action),
       return what is between that marker and the next ```.
       → works even if the LLM writes other text or code blocks before it
         (e.g. its step-by-step thoughts).
    2. Otherwise, fall back to the first code block of any type.
    3. If there is no code block at all, return the response unchanged.
    """
    marker = f"```{block_type}"

    # 1. Look for the block with the right tag, e.g. "```action"
    if marker in response:
        after_marker = response.split(marker, 1)[1]
        return after_marker.split("```", 1)[0].strip()

    # 3. No code block at all
    if '```' not in response:
        return response

    # 2. Fallback: first code block, without its language tag
    code_block = response.split('```')[1].strip()
    if code_block.startswith(block_type):
        code_block = code_block[len(block_type):].strip()
    return code_block


def parse_action(response: str) -> Dict:
    """Parse the LLM response into a structured action dictionary.

    Returns:
        {"tool_name": ..., "args": {...}} if the response is a valid tool call,
        otherwise an "error" action whose message tells the LLM how to fix it.
    """
    try:
        # 1. Keep only the content of the ```action block
        response = extract_markdown_block(response, "action")
        # 2. Text -> Python dict
        response_json = json.loads(response)
        # 3. Check that it is a proper tool invocation
        if "tool_name" in response_json and "args" in response_json:
            return response_json
        else:
            return {"tool_name": "error", "args": {"message": "You must respond with a JSON tool invocation."}}
    except json.JSONDecodeError:
        # Not valid JSON at all
        return {"tool_name": "error", "args": {"message": "Invalid JSON response. You must respond with a JSON tool invocation."}}

In [ ]:
# Step 3: parse the response generated in step 2
action = parse_action(response)
print(action)

## Step 4 · Execute the action

**Goal:** run the Python function that matches the parsed `tool_name`, with the parsed `args`.

- Each tool in the agent rules ↔ one function in the code.
- Mapping `tool_name` → function:

| `tool_name` | What happens | Result |
|---|---|---|
| `list_files` | call `list_files()` | `{"result": [files...]}` |
| `read_file` | call `read_file(args["file_name"])` | `{"result": file content}` |
| `error` | (from the parser) | `{"error": message}` |
| `terminate` | print the final message | stop the loop (`break`) |
| anything else | unknown tool | `{"error": "Unknown action: ..."}` |

- This is where the agent does **tangible work** (reads files, prints messages).
- → bridges the **decision** (LLM) with **concrete results**, which go back into memory for the next iteration.

First, the tools themselves:

In [ ]:
import os

# --- Tools: the real functions behind the names in agent_rules ---

def list_files() -> List[str]:
    """List all files in the current directory."""
    return os.listdir(".")


def read_file(file_name: str) -> str:
    """Read the content of a file (or return an error message)."""
    try:
        with open(file_name, "r") as file:
            return file.read()
    except FileNotFoundError:
        return f"Error: {file_name} not found."
    except Exception as e:
        return f"Error: {str(e)}"

Then the execution logic. In the full agent this sits **inside the loop**, where `terminate` uses `break` to exit; since the loop is not built yet, here `terminate` just prints the message and stores it.

In [ ]:
# Step 4: map tool_name -> function and run it with the parsed args
if action["tool_name"] == "list_files":
    result = {"result": list_files()}
elif action["tool_name"] == "read_file":
    result = {"result": read_file(action["args"]["file_name"])}
elif action["tool_name"] == "error":
    result = {"error": action["args"]["message"]}
elif action["tool_name"] == "terminate":
    print(action["args"]["message"])
    result = {"terminate": action["args"]["message"]}   # in the loop: break
else:
    result = {"error": "Unknown action: " + action["tool_name"]}

print(result)

## Step 5 · Update the memory

**Goal:** record what just happened, so the next iteration can build on it.

- Memory = the agent's record of the interaction:
  - user requests,
  - actions performed,
  - their outcomes.
- After each action, **two messages** are appended:

| Role | Content | Meaning |
|---|---|---|
| `assistant` | the LLM's `response` | the agent's **intention** (the action it chose) |
| `user` | `json.dumps(result)` | the **feedback**: what happened when the action ran |

- Results of actions are **always** sent back with the `user` role.
- Growing history → continuity → the agent refines its behaviour and tracks the status of its work.

In [ ]:
# Step 5: add the agent's action and its result to the memory
memory.extend([
    {"role": "assistant", "content": response},          # what the agent decided
    {"role": "user", "content": json.dumps(result)},      # what happened (as text)
])

# Show the updated memory
for m in memory:
    print(f"{m['role']:>9}: {m['content']}")

## Step 6 · Decide whether to continue

**Goal:** at the end of each iteration, decide → **loop again** or **stop**.

- Stop if:
  - the action is `terminate` → print its closing message and exit the loop (`break`);
  - a predefined limit is reached, e.g. **maximum number of iterations** (safety net against endless loops).
- Otherwise → back to step 1 with the updated memory (next user request, or next part of the task).

```python
if action["tool_name"] == "terminate":
    print(action["args"]["message"])
    break
```

**Example: iterative adaptation** (task: *read a file*, but no file name given)

1. Iteration 1 → `list_files` → gets the available files.
2. Iteration 2 → using that result from memory → asks the user to pick a file.
3. … continues until the task is done → `terminate`.

- **Memory is what decides when to stop**: at each step the agent looks back to check whether the overall task is complete.
- Balance: adapt to new information ↔ eventually conclude.
- Stopping rules can also be **instructed** in the agent rules (e.g. *stop after more than two errors*).

## Improved agent rules

A more robust version of the system message from Step 1, used by the full loop below.

| | Step 1 rules | Improved rules |
|---|---|---|
| **Tools** | one line each: `list_files() -> List[str]: ...` | **JSON schema**: `description` + `parameters` (name, `type`, `description`) |
| **Reading files** | *list them before reading* | *if asked about files, documents or content → list first, then read* |
| **Ending** | `terminate` only listed as a tool | explicit: *when done, use `terminate`* |
| **Response format** | action block only | **think step by step first**, then the action block |
| **Emphasis** | *Every response MUST have an action* | *Important!!! … You must ALWAYS respond in this format* |

Why these changes:

- **JSON tool schema** → the same structured format real LLM APIs use for tools ("function calling"); parameter names and types are unambiguous → *parameters map to `args`*.
- **Think step by step** before acting → the model plans its move (chain of thought) → better choice of tool and arguments.
  - The thoughts are plain text **before** the ```` ```action ```` block → `extract_markdown_block` now looks specifically for the ```` ```action ```` block, so the thoughts (or a ```` ```json ```` block in them) don't break parsing.
- **Explicit termination** → the agent knows how and when to finish instead of looping until `max_iterations`.

In [ ]:
# Improved agent rules: tools described as a JSON schema + step-by-step thinking
## Agent prompt: Goals → role and task, Actions → the tool list, Information → memory, Language → the response format
agent_rules = [{
    "role": "system",
    "content": """
You are an AI agent that can perform tasks by using available tools.

Available tools:

```json
{
    "list_files": {
        "description": "Lists all files in the current directory.",
        "parameters": {}
    },
    "read_file": {
        "description": "Reads the content of a file.",
        "parameters": {
            "file_name": {
                "type": "string",
                "description": "The name of the file to read."
            }
        }
    },
    "terminate": {
        "description": "Ends the agent loop and provides a summary of the task.",
        "parameters": {
            "message": {
                "type": "string",
                "description": "Summary message to return to the user."
            }
        }
    }
}
```

If a user asks about files, documents, or content, first list the files before reading them.

When you are done, terminate the conversation by using the "terminate" tool and I will provide the results to the user.

Important!!! Every response MUST have an action.
You must ALWAYS respond in this format:

<Stop and think step by step. Parameters map to args. Insert a rich description of your step by step thoughts here.>

```action
{
    "tool_name": "insert tool_name",
    "args": {...fill in any required arguments here...}
}
```"""
}]

## Putting it all together: the agent loop

All six steps in one `while` loop, using the **improved agent rules** above. Here `terminate` really uses `break`, and `max_iterations` stops the agent if it never terminates.

Try for example: *"What files are in this directory? Read the README and summarise it."*

In [ ]:
max_iterations = 10   # safety net: stop even if the agent never calls terminate
iterations = 0

# The task comes from the user and is the first message in memory
user_task = input("What would you like me to do? ")
memory = [{"role": "user", "content": user_task}]

while iterations < max_iterations:
    # 1. Construct prompt: rules + memory
    prompt = agent_rules + memory

    # 2. Generate response: the LLM chooses the next action
    print("Agent thinking...")
    response = generate_response(prompt)
    print(f"Agent response: {response}")

    # 3. Parse response -> {"tool_name": ..., "args": ...}
    action = parse_action(response)

    # 4. Execute action  (+ 6. stop on terminate)
    if action["tool_name"] == "list_files":
        result = {"result": list_files()}
    elif action["tool_name"] == "read_file":
        result = {"result": read_file(action["args"]["file_name"])}
    elif action["tool_name"] == "error":
        result = {"error": action["args"]["message"]}
    elif action["tool_name"] == "terminate":
        print(action["args"]["message"])
        break
    else:
        result = {"error": "Unknown action: " + action["tool_name"]}

    print(f"Action result: {result}")

    # 5. Update memory with the action and its result
    memory.extend([
        {"role": "assistant", "content": response},
        {"role": "user", "content": json.dumps(result)},
    ])

    # 6. Continue? (terminate handled above; here: count iterations)
    iterations += 1
else:
    print(f"Stopped: reached the maximum of {max_iterations} iterations.")